# Multi-Class Animal Detection: retrain and test-set evaluation

Sem V minor project (LNCT, Jul–Dec 2024), team of four: Aashman Singh Verma, Abhishree Soni, Khanak Agrawal, Parth Nagar.

The 2024 run's fine-tuned weights were never saved (the `best_model.pt` kept from 2024 is the stock COCO `yolov8n.pt`), and the report never recorded a result. This notebook re-runs the training with **the settings written in the 2024 report** and measures the result on the **held-out test split**, which training never sees.

**Before you press Run All (Kaggle side panel → Settings):**
1. Accelerator: **GPU T4 x2** (the notebook trains on one of the two GPUs via `DEVICE = 0`; single-GPU training is simpler and more reliable in a notebook)
2. Internet: **On** (needed for `pip install ultralytics` and the pretrained `yolov8n.pt`)
3. Input → Add Input → your dataset **notaashman/multiclass-animal-detection**

Expected run time: roughly 15–25 minutes on one GPU.

## 1. Settings
Everything the run depends on, in one place. The training values come from the 2024 report (Feature Extraction §2–4).

In [ ]:
SEARCH_ROOT = "/kaggle/input"      # where Kaggle mounts attached datasets
WORK_DIR    = "/kaggle/working"    # the only writable folder on Kaggle

EPOCHS = 50
BATCH  = 16
IMGSZ  = 640
SEED   = 0          # fixed so the run can be repeated
DEVICE = 0          # first GPU; use "cpu" only for a quick test

# Augmentation, as written in the 2024 report
AUG = dict(
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,   # colour: hue / saturation / brightness
    translate=0.1,                       # shift up to ±10%
    scale=0.5,                           # zoom up to ±50%
    shear=2.0,                           # shear in degrees
    fliplr=0.5, flipud=0.0,              # horizontal flip half the time, never vertical
)

## 2. Install and check the environment

In [ ]:
!pip install -q ultralytics

import os, glob, json, shutil, platform
import torch, ultralytics
from ultralytics import YOLO

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| python", platform.python_version())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)")

## 3. Find the dataset and check the split

Roboflow's `data.yaml` uses relative paths (`../train/images`) that don't resolve on Kaggle, and `/kaggle/input` is read-only. So this cell finds the dataset, checks the split is the expected **1,400 / 300 / 300**, and writes a corrected copy of `data.yaml` into the working folder.

In [ ]:
import yaml

found = glob.glob(os.path.join(SEARCH_ROOT, "**", "data.yaml"), recursive=True)
assert len(found) == 1, f"Expected exactly one data.yaml under {SEARCH_ROOT}, found: {found}"
DATA_ROOT = os.path.dirname(found[0])
print("Dataset folder:", DATA_ROOT)

with open(found[0]) as f:
    src = yaml.safe_load(f)
CLASS_NAMES = src["names"]
assert src["nc"] == 20 and len(CLASS_NAMES) == 20, "Expected 20 classes"

IMG_EXT = (".jpg", ".jpeg", ".png")
split_counts = {}
for split in ["train", "valid", "test"]:
    imgs = [p for p in os.listdir(os.path.join(DATA_ROOT, split, "images")) if p.lower().endswith(IMG_EXT)]
    lbls = [p for p in os.listdir(os.path.join(DATA_ROOT, split, "labels")) if p.endswith(".txt")]
    boxes = 0
    for p in lbls:
        with open(os.path.join(DATA_ROOT, split, "labels", p)) as f:
            boxes += sum(1 for line in f if line.strip())
    split_counts[split] = {"images": len(imgs), "label_files": len(lbls), "boxes": boxes}
    print(f"{split:5s}: {len(imgs):4d} images, {len(lbls):4d} label files, {boxes:4d} boxes")

data_cfg = {
    "path": DATA_ROOT,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 20,
    "names": CLASS_NAMES,
}
DATA_YAML = os.path.join(WORK_DIR, "data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print("\nWrote", DATA_YAML)

## 4. Train

Starts from COCO-pretrained `yolov8n.pt` (transfer learning) and fine-tunes on the 20 animal classes. During training, Ultralytics checks the model on the **validation** split after each epoch and keeps the best epoch as `best.pt`.

In [ ]:
model = YOLO("yolov8n.pt")

train_results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    seed=SEED,
    deterministic=True,
    device=DEVICE,
    project=os.path.join(WORK_DIR, "runs"),
    name="train",
    exist_ok=True,
    **AUG,
)
TRAIN_DIR = str(train_results.save_dir) if train_results is not None else os.path.join(WORK_DIR, "runs", "train")
BEST = os.path.join(TRAIN_DIR, "weights", "best.pt")
print("Best weights:", BEST)

## 5. Sanity check: is this really the fine-tuned model?

The 2024 mistake was keeping the stock COCO model by accident. This cell refuses to go on unless `best.pt` knows exactly the 20 animal classes.

In [ ]:
best = YOLO(BEST)
model_names = [best.names[i] for i in range(len(best.names))]
assert model_names == CLASS_NAMES, f"best.pt has the wrong classes: {model_names[:5]}..."
print("OK: best.pt has the 20 animal classes, not COCO's 80.")

## 6. Evaluate on the test split

**This is the number to report.** The test images were never used for training or for choosing the best epoch. Confidence threshold is Ultralytics' evaluation default (0.001), which is the standard way to compute mAP; the demo app's 0.4 threshold is only for display.

In [ ]:
def evaluate(split):
    m = best.val(
        data=DATA_YAML, split=split, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
        project=os.path.join(WORK_DIR, "runs"), name=f"eval_{split}", exist_ok=True, plots=True,
    )
    overall = {
        "precision": float(m.box.mp),
        "recall": float(m.box.mr),
        "mAP50": float(m.box.map50),
        "mAP50_95": float(m.box.map),
    }
    per_class = []
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        per_class.append({"class": CLASS_NAMES[int(c)], "precision": float(p), "recall": float(r),
                          "AP50": float(ap50), "AP50_95": float(ap)})
    return overall, per_class, str(m.save_dir)

test_overall, test_per_class, TEST_DIR = evaluate("test")
val_overall,  _,              VAL_DIR  = evaluate("val")

print("\nTEST split  (", split_counts["test"]["images"], "images,", split_counts["test"]["boxes"], "boxes )")
for k, v in test_overall.items():
    print(f"  {k:10s} {v:.3f}")
print("\nValidation split (used to pick the best epoch, so slightly optimistic)")
for k, v in val_overall.items():
    print(f"  {k:10s} {v:.3f}")

In [ ]:
import pandas as pd
per_class_df = pd.DataFrame(test_per_class).sort_values("AP50").reset_index(drop=True)
per_class_df.round(3)   # weakest classes at the top

## 7. Save everything for the GitHub repo
All of it lands in `/kaggle/working/export/`. After **Save Version → Save & Run All**, it appears under the notebook's **Output** tab, where you can download it.

In [ ]:
EXPORT = os.path.join(WORK_DIR, "export")
os.makedirs(EXPORT, exist_ok=True)

shutil.copy(BEST, os.path.join(EXPORT, "best.pt"))
shutil.copy(DATA_YAML, os.path.join(EXPORT, "data.yaml"))
for name in ["results.csv", "results.png", "args.yaml"]:
    p = os.path.join(TRAIN_DIR, name)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(EXPORT, name))
for p in glob.glob(os.path.join(TEST_DIR, "*.png")):
    shutil.copy(p, os.path.join(EXPORT, "test_" + os.path.basename(p)))

per_class_df.to_csv(os.path.join(EXPORT, "test_per_class.csv"), index=False)

record = {
    "what": "YOLOv8n fine-tuned on Multi-Class Animal Detection (20 classes), retrained by Aashman Singh Verma",
    "dataset_split": split_counts,
    "settings": {"epochs": EPOCHS, "batch": BATCH, "imgsz": IMGSZ, "seed": SEED, **AUG},
    "environment": {"ultralytics": ultralytics.__version__, "torch": torch.__version__,
                    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"},
    "test_metrics": test_overall,
    "val_metrics": val_overall,
}
with open(os.path.join(EXPORT, "metrics.json"), "w") as f:
    json.dump(record, f, indent=2)

print(json.dumps(record["test_metrics"], indent=2))
print("\nExported:", sorted(os.listdir(EXPORT)))

## Outputs

After **Save Version → Save & Run All**, everything in `/kaggle/working/export/` appears under the notebook's **Output** tab: `best.pt`, `metrics.json`, `results.csv`, `test_per_class.csv`, `args.yaml` and the plots. Those are the files in this repository's `results/` folder.